# 📰 News Scraper — Trial Run
**Purpose:** Scrape open-access RSS feeds from think-tanks and wire services,
produce structured JSONL output, and run frequency analysis for indexation and prediction pipelines.

**No file uploads needed.** All source files are written directly into Colab's filesystem by the setup cells.

Run cells **top to bottom**. Each section is independent.

| Section | What it does |
|---|---|
| 1 · Setup | Install deps + write all source files inline |
| 2 · Config | Review / edit keywords and feeds |
| 3 · Scrape | Run the scraper |
| 4 · Inspect | Browse raw output |
| 5 · Analyse | Frequency analysis across corpus |
| 6 · Enrich | Per-article `analysis{}` block |
| 7 · Export | Download files to local machine |


---
## 1 · Setup


In [ ]:
# 1.1 — Install dependencies
!pip install -q feedparser newspaper3k requests beautifulsoup4 lxml lxml_html_clean langdetect

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ Dependencies installed')


In [ ]:
# 1.2 — Create the news_scraper/ package directory
import os
os.makedirs('/content/news_scraper', exist_ok=True)
print('✅ /content/news_scraper/ ready')


#### Write `config.py`


In [ ]:
# Auto-generated: writes config.py into the Colab filesystem
_src = """\"\"\"
news_scraper/config.py
----------------------
All scraper settings live here — load from JSON/YAML or override programmatically.
\"\"\"

from __future__ import annotations
import json
from dataclasses import dataclass, field, asdict
from pathlib import Path


@dataclass
class ScraperConfig:
    # ── What to look for ──────────────────────────────────────────
    keywords: list[str] = field(default_factory=lambda: [
        "war", "Iran", "IRGC", "US", "Israel",
        "missile", "sanctions", "nuclear", "ceasefire", "airstrike",
    ])

    # ── Where to look ─────────────────────────────────────────────
    feeds: list[str] = field(default_factory=lambda: [

        # ── US think-tanks & policy institutes (all open access) ──
        "https://www.csis.org/rss.xml",                          # CSIS
        "https://www.rand.org/pubs/commentary.xml",              # RAND Commentary
        "https://www.rand.org/pubs/research_reports.xml",        # RAND Research Reports
        "https://carnegieendowment.org/rss",                     # Carnegie Endowment
        "https://www.brookings.edu/feed/?post_type=research",    # Brookings Institution

        # ── UK / European think-tanks ─────────────────────────────
        "https://www.chathamhouse.org/rss.xml",                  # Chatham House (RIIA)
        "https://www.iiss.org/rss",                              # IISS — military & strategic affairs

        # ── Foreign policy & security journals ────────────────────
        "https://foreignaffairs.com/rss.xml",                    # Foreign Affairs (CFR) — free articles
        "https://foreignpolicy.com/feed",                        # Foreign Policy magazine
        "https://warontherocks.com/feed",                        # War on the Rocks — defence analysis
        "https://iswresearch.blogspot.com/feeds/posts/default",  # ISW — Institute for the Study of War

        # ── Anglophone wire / broadcast ───────────────────────────
        "https://feeds.bbci.co.uk/news/world/rss.xml",          # BBC World
        "https://feeds.reuters.com/reuters/worldNews",           # Reuters World
        "https://apnews.com/rss/apf-intlnews",                   # Associated Press
        "https://www.theguardian.com/world/rss",                 # The Guardian

        # ── Regional / Global South ───────────────────────────────
        "https://www.aljazeera.com/xml/rss/all.xml",             # Al Jazeera
        "https://timesofindia.indiatimes.com/rssfeeds/296589292.cms", # Times of India
        "https://www.france24.com/en/rss",                       # France 24 (EN)
        "https://feeds.dw.com/rw/rss/rss.xml",                   # Deutsche Welle

        # ── Defence / security specialist ─────────────────────────
        "https://www.spacewar.com/Military_Technology.xml",      # Spacewar — defence tech
        "https://www.defense.gov/DesktopModules/ArticleCS/RSS.ashx?ContentType=1&Site=945&max=10",  # US DoD News
    ])

    # ── Crawl behaviour ───────────────────────────────────────────
    max_entries_per_feed:    int   = 50
    delay_between_requests:  float = 2.0   # seconds, per-article within a feed
    request_timeout:         int   = 15    # seconds
    max_workers:             int   = 4     # parallel feed fetchers

    # ── Output ────────────────────────────────────────────────────
    output_dir:     str = "output"
    save_json:      bool = True
    save_jsonl:     bool = True   # one article per line — great for streaming / ML
    save_index:     bool = True   # lightweight index file for fast lookups
    min_word_count: int  = 80     # drop stub articles

    # ── Prediction helpers ────────────────────────────────────────
    # Entity categories for downstream NER tagging hints
    entity_hints: dict[str, list[str]] = field(default_factory=lambda: {
        "countries":       ["Iran", "Israel", "US", "USA", "United States", "Russia", "China"],
        "organizations":   ["IRGC", "NATO", "UN", "CIA", "Mossad", "IDF", "Pentagon"],
        "event_types":     ["airstrike", "ceasefire", "sanctions", "nuclear", "missile", "invasion"],
    })

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_json(cls, path: str | Path) -> "ScraperConfig":
        with open(path) as f:
            data = json.load(f)
        return cls(**data)

    def save_json_file(self, path: str | Path) -> None:
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)
"""
with open('/content/news_scraper/config.py', 'w', encoding='utf-8') as _f:
    _f.write(_src)
print('wrote config.py')


#### Write `models.py`


In [ ]:
# Auto-generated: writes models.py into the Colab filesystem
_src = """\"\"\"
news_scraper/models.py
----------------------
Pure-Python dataclasses that form the schema for every article and session.
Designed to map 1-to-1 with a search index (Elasticsearch, Typesense, etc.)
or a vector store (Pinecone, Weaviate, Chroma).
\"\"\"

from __future__ import annotations
from dataclasses import dataclass, field, asdict
from typing import Optional


# ─────────────────────────────────────────────
#  Core article record — the indexation unit
# ─────────────────────────────────────────────

@dataclass
class ArticleRecord:
    # ── Identity ──────────────────────────────────────────────────
    url_hash:   str = ""          # SHA-256 of URL — dedup key
    url:        str = ""
    source_feed: str = ""
    scraped_at: str = ""          # ISO-8601 UTC

    # ── Content ───────────────────────────────────────────────────
    title:      str = ""
    authors:    list[str] = field(default_factory=list)
    publish_date: Optional[str] = None   # ISO-8601 or None
    full_text:  str = ""
    summary:    str = ""          # newspaper3k NLP summary
    top_image:  str = ""

    # ── NLP / ML features ─────────────────────────────────────────
    extracted_keywords: list[str] = field(default_factory=list)   # newspaper3k
    matched_keywords:   list[str] = field(default_factory=list)   # our keyword set
    relevance_score:    float = 0.0    # 0–1, higher = more keyword hits
    word_count:         int   = 0
    language:           str   = "unknown"

    # ── Prediction placeholders ───────────────────────────────────
    # These fields are intentionally empty at scrape time — downstream
    # NLP / ML models should populate them.
    entities:           dict  = field(default_factory=dict)
    # e.g. {"countries": ["Iran", "Israel"], "organizations": ["IRGC"]}

    sentiment:          dict  = field(default_factory=dict)
    # e.g. {"label": "negative", "score": 0.82}

    topic_tags:         list[str] = field(default_factory=list)
    # e.g. ["military-conflict", "diplomacy", "sanctions"]

    embedding_model:    str   = ""    # name of model used to embed
    embedding_vector:   list  = field(default_factory=list)  # float32[]

    prediction:         dict  = field(default_factory=dict)
    # e.g. {"escalation_risk": 0.74, "conflict_class": "armed-conflict"}

    def to_dict(self) -> dict:
        return asdict(self)

    def to_index_record(self) -> dict:
        \"\"\"Lightweight dict for search index (drops embedding vector).\"\"\"
        d = self.to_dict()
        d.pop("embedding_vector", None)
        d.pop("full_text", None)
        return d

    def to_jsonl_line(self) -> str:
        import json
        return json.dumps(self.to_dict(), ensure_ascii=False)


# ─────────────────────────────────────────────
#  Per-feed result wrapper
# ─────────────────────────────────────────────

@dataclass
class FeedResult:
    feed_url: str = ""
    articles: list[ArticleRecord] = field(default_factory=list)
    error:    Optional[str] = None

    @property
    def count(self) -> int:
        return len(self.articles)


# ─────────────────────────────────────────────
#  Session — one full scrape run
# ─────────────────────────────────────────────

@dataclass
class ScrapeSession:
    session_id:      str = ""
    started_at:      str = ""
    finished_at:     str = ""
    total_scraped:   int = 0
    config_snapshot: dict = field(default_factory=dict)

    articles:        list[ArticleRecord] = field(default_factory=list)
    feed_results:    list[FeedResult]    = field(default_factory=list)

    def summary(self) -> dict:
        by_feed = {fr.feed_url: fr.count for fr in self.feed_results}
        top = sorted(self.articles, key=lambda a: a.relevance_score, reverse=True)[:5]
        return {
            "session_id":    self.session_id,
            "started_at":    self.started_at,
            "finished_at":   self.finished_at,
            "total_articles": self.total_scraped,
            "articles_by_feed": by_feed,
            "top_articles":  [
                {"title": a.title, "score": a.relevance_score, "url": a.url}
                for a in top
            ],
            "errors": [
                {"feed": fr.feed_url, "error": fr.error}
                for fr in self.feed_results if fr.error
            ],
        }
"""
with open('/content/news_scraper/models.py', 'w', encoding='utf-8') as _f:
    _f.write(_src)
print('wrote models.py')


#### Write `scraper.py`


In [ ]:
# Auto-generated: writes scraper.py into the Colab filesystem
_src = """\"\"\"
news_scraper/scraper.py
-----------------------
Enhanced news scraper with structured output for indexation and prediction pipelines.
\"\"\"
from __future__ import annotations

import feedparser
import requests
from newspaper import Article
import time
import json
import hashlib
import logging
import re
import sys
import os
import nltk
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Optional
from bs4 import BeautifulSoup

# ── robust import: works in scripts, notebooks, and Colab ──────────────
def _colab_import():
    import importlib, sys, os
    pkg = os.path.dirname(os.path.abspath(__file__))
    if pkg not in sys.path:
        sys.path.insert(0, pkg)
    for mod in ['config', 'models']:
        if mod not in sys.modules:
            importlib.import_module(mod)

_colab_import()

from config import ScraperConfig
from models import ArticleRecord, FeedResult, ScrapeSession

# ── nltk data required by newspaper3k .nlp() ──────────────────────────
def _ensure_nltk_data() -> None:
    for pkg in ("punkt", "punkt_tab"):
        try:
            nltk.data.find(f"tokenizers/{pkg}")
        except LookupError:
            nltk.download(pkg, quiet=True)

logger = logging.getLogger(__name__)


# ─────────────────────────────────────────────
#  Keyword matching
# ─────────────────────────────────────────────

def score_article(text: str, title: str, keywords: list[str]) -> tuple[bool, list[str], float]:
    \"\"\"
    Returns (matches, matched_keywords, relevance_score).
    Score = weighted hit count / total keywords (title hits worth 2x).
    \"\"\"
    matched = set()
    title_lower = (title or "").lower()
    text_lower  = (text  or "").lower()

    for kw in keywords:
        kw_lower = kw.lower()
        in_title = kw_lower in title_lower
        in_text  = kw_lower in text_lower
        if in_title or in_text:
            matched.add(kw)

    if not matched:
        return False, [], 0.0

    title_hits = sum(1 for kw in matched if kw.lower() in title_lower)
    text_hits  = len(matched) - title_hits
    raw_score  = (title_hits * 2 + text_hits) / (len(keywords) * 2)
    score      = round(min(raw_score, 1.0), 4)
    return True, sorted(matched), score


# ─────────────────────────────────────────────
#  Article extraction
# ─────────────────────────────────────────────

def _extract_with_newspaper(url: str, timeout: int) -> Optional[dict]:
    _ensure_nltk_data()
    art = Article(url, request_timeout=timeout)
    art.download()
    art.parse()
    art.nlp()
    return {
        "title":    art.title,
        "text":     art.text,
        "authors":  art.authors,
        "summary":  art.summary,
        "keywords": art.keywords,
        "top_image": art.top_image,
        "publish_date": art.publish_date,
    }


def _extract_with_bs4(url: str, timeout: int) -> Optional[dict]:
    \"\"\"Fallback extractor using requests + BeautifulSoup.\"\"\"
    resp = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    title = soup.find("title")
    title_text = title.get_text(strip=True) if title else ""

    # grab <p> tags as body text
    paragraphs = soup.find_all("p")
    body = " ".join(p.get_text(separator=" ", strip=True) for p in paragraphs)
    body = re.sub(r"\\s{2,}", " ", body).strip()

    return {
        "title":       title_text,
        "text":        body,
        "authors":     [],
        "summary":     body[:500] if body else "",
        "keywords":    [],
        "top_image":   "",
        "publish_date": None,
    }


def get_article_text(url: str, timeout: int = 15) -> Optional[dict]:
    for extractor in (_extract_with_newspaper, _extract_with_bs4):
        try:
            data = extractor(url, timeout)
            if data and data.get("text"):
                return data
        except Exception as e:
            logger.debug("Extractor %s failed for %s: %s", extractor.__name__, url, e)
    logger.warning("All extractors failed for %s", url)
    return None


# ─────────────────────────────────────────────
#  Feed parsing
# ─────────────────────────────────────────────

def _parse_single_feed(feed_url: str, cfg: ScraperConfig) -> FeedResult:
    result = FeedResult(feed_url=feed_url)
    seen_in_feed: set[str] = set()

    try:
        feed = feedparser.parse(feed_url)
        entries = feed.entries[: cfg.max_entries_per_feed]
        logger.info("Feed %s — %d entries", feed_url, len(entries))

        for entry in entries:
            title   = entry.get("title", "")
            link    = entry.get("link", "")
            summary = entry.get("summary", "")

            if not link:
                continue

            url_hash = hashlib.sha256(link.encode()).hexdigest()
            if url_hash in seen_in_feed:
                continue
            seen_in_feed.add(url_hash)

            matches, matched_kws, score = score_article(
                summary, title, cfg.keywords
            )
            if not matches:
                continue

            logger.info("  Matched: %s (score=%.2f)", title[:80], score)
            article_data = get_article_text(link, timeout=cfg.request_timeout)

            if article_data:
                # re-score with full text
                _, matched_kws, score = score_article(
                    article_data["text"], title, cfg.keywords
                )

                pub_date = article_data.get("publish_date")
                if isinstance(pub_date, datetime):
                    pub_date = pub_date.isoformat()

                record = ArticleRecord(
                    url_hash        = url_hash,
                    url             = link,
                    source_feed     = feed_url,
                    scraped_at      = datetime.now(timezone.utc).isoformat(),
                    title           = article_data["title"] or title,
                    authors         = article_data["authors"],
                    publish_date    = pub_date,
                    full_text       = article_data["text"],
                    summary         = article_data["summary"],
                    extracted_keywords = article_data["keywords"],
                    top_image       = article_data.get("top_image", ""),
                    matched_keywords = matched_kws,
                    relevance_score = score,
                    word_count      = len(article_data["text"].split()),
                    language        = _detect_language(article_data["text"]),
                )
                result.articles.append(record)
                time.sleep(cfg.delay_between_requests)

    except Exception as e:
        logger.error("Feed parse error [%s]: %s", feed_url, e)
        result.error = str(e)

    return result


def _detect_language(text: str) -> str:
    \"\"\"Naive language tag — extend with langdetect if needed.\"\"\"
    try:
        from langdetect import detect
        return detect(text[:500])
    except Exception:
        return "unknown"


# ─────────────────────────────────────────────
#  Main scrape orchestrator
# ─────────────────────────────────────────────

def scrape(cfg: ScraperConfig) -> ScrapeSession:
    session = ScrapeSession(
        session_id  = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"),
        started_at  = datetime.now(timezone.utc).isoformat(),
        config_snapshot = cfg.to_dict(),
    )

    global_seen: set[str] = set()

    with ThreadPoolExecutor(max_workers=cfg.max_workers) as pool:
        futures = {
            pool.submit(_parse_single_feed, url, cfg): url
            for url in cfg.feeds
        }
        for future in as_completed(futures):
            feed_result = future.result()
            session.feed_results.append(feed_result)

            for article in feed_result.articles:
                if article.url_hash not in global_seen:
                    global_seen.add(article.url_hash)
                    session.articles.append(article)
                else:
                    logger.debug("Duplicate skipped: %s", article.url)

    session.finished_at   = datetime.now(timezone.utc).isoformat()
    session.total_scraped = len(session.articles)
    return session
"""
with open('/content/news_scraper/scraper.py', 'w', encoding='utf-8') as _f:
    _f.write(_src)
print('wrote scraper.py')


#### Write `writer.py`


In [ ]:
# Auto-generated: writes writer.py into the Colab filesystem
_src = """\"\"\"
news_scraper/writer.py
----------------------
Handles all output: structured JSON, JSONL corpus, and a lightweight index.
\"\"\"
from __future__ import annotations

import json
import logging
import sys
import os
from pathlib import Path
from datetime import datetime, timezone

def _colab_import():
    import importlib, sys, os
    pkg = os.path.dirname(os.path.abspath(__file__))
    if pkg not in sys.path:
        sys.path.insert(0, pkg)
    for mod in ['config', 'models']:
        if mod not in sys.modules:
            importlib.import_module(mod)

_colab_import()

from config import ScraperConfig
from models import ScrapeSession

logger = logging.getLogger(__name__)


def write_outputs(session: ScrapeSession, cfg: ScraperConfig) -> dict[str, Path]:
    \"\"\"
    Writes all configured output files and returns a dict of {type: path}.

    Output structure:
        output/
          sessions/
            <session_id>/
              session_meta.json      ← summary + config snapshot
              articles_full.json     ← all ArticleRecord dicts (with full text)
              articles.jsonl         ← one JSON line per article (ML-friendly)
          index/
              master_index.jsonl     ← cumulative lightweight index (appended)
    \"\"\"
    out_root = Path(cfg.output_dir)
    session_dir = out_root / "sessions" / session.session_id
    index_dir   = out_root / "index"

    session_dir.mkdir(parents=True, exist_ok=True)
    index_dir.mkdir(parents=True, exist_ok=True)

    written: dict[str, Path] = {}

    # ── Filter stubs ──────────────────────────────────────────────
    articles = [
        a for a in session.articles
        if a.word_count >= cfg.min_word_count
    ]
    logger.info("Writing %d articles (filtered from %d)", len(articles), len(session.articles))

    # ── session_meta.json ─────────────────────────────────────────
    meta_path = session_dir / "session_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(session.summary(), f, indent=2, ensure_ascii=False, default=str)
    written["meta"] = meta_path
    logger.info("Saved session meta → %s", meta_path)

    # ── articles_full.json ────────────────────────────────────────
    if cfg.save_json:
        full_path = session_dir / "articles_full.json"
        payload = {
            "session_id": session.session_id,
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total": len(articles),
            "articles": [a.to_dict() for a in articles],
        }
        with open(full_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
        written["json"] = full_path
        logger.info("Saved full JSON → %s", full_path)

    # ── articles.jsonl ────────────────────────────────────────────
    if cfg.save_jsonl:
        jsonl_path = session_dir / "articles.jsonl"
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for a in articles:
                f.write(a.to_jsonl_line() + "\\n")
        written["jsonl"] = jsonl_path
        logger.info("Saved JSONL corpus → %s", jsonl_path)

    # ── master_index.jsonl (append) ───────────────────────────────
    if cfg.save_index:
        index_path = index_dir / "master_index.jsonl"
        with open(index_path, "a", encoding="utf-8") as f:
            for a in articles:
                f.write(json.dumps(a.to_index_record(), ensure_ascii=False, default=str) + "\\n")
        written["index"] = index_path
        logger.info("Appended to master index → %s", index_path)

    return written
"""
with open('/content/news_scraper/writer.py', 'w', encoding='utf-8') as _f:
    _f.write(_src)
print('wrote writer.py')


#### Write `__main__.py`


In [ ]:
# Auto-generated: writes __main__.py into the Colab filesystem
_src = """\"\"\"
news_scraper/__main__.py
------------------------
CLI entry point — also importable as a function for Colab / notebook use.

Usage (terminal):
    python -m news_scraper
    python -m news_scraper --config my_config.json
    python -m news_scraper --keywords war Iran --feeds https://...

Usage (Colab / notebook):
    from news_scraper.__main__ import run
    run()
\"\"\"
from __future__ import annotations

import argparse
import logging
import sys
import os
from pathlib import Path

def _colab_import():
    import importlib, sys, os
    pkg = os.path.dirname(os.path.abspath(__file__))
    if pkg not in sys.path:
        sys.path.insert(0, pkg)
    for mod in ['config', 'models', 'scraper', 'writer']:
        if mod not in sys.modules:
            importlib.import_module(mod)

_colab_import()

from config import ScraperConfig
from scraper import scrape
from writer import write_outputs


# ─────────────────────────────────────────────
#  Logging setup
# ─────────────────────────────────────────────

def setup_logging(verbose: bool = False, log_dir: str = ".") -> None:
    level = logging.DEBUG if verbose else logging.INFO
    fmt   = "%(asctime)s  %(levelname)-8s  %(name)s — %(message)s"
    log_path = Path(log_dir) / "scraper.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    # avoid duplicate handlers when re-run in a notebook
    root = logging.getLogger()
    if root.handlers:
        root.handlers.clear()
    logging.basicConfig(
        level=level,
        format=fmt,
        handlers=[
            logging.StreamHandler(sys.stdout),
            logging.FileHandler(str(log_path), encoding="utf-8"),
        ],
    )


# ─────────────────────────────────────────────
#  CLI
# ─────────────────────────────────────────────

def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(
        description="Structured news scraper for indexation and prediction pipelines."
    )
    p.add_argument("--config",    type=str, help="Path to JSON config file.")
    p.add_argument("--keywords",  nargs="+", help="Override keywords list.")
    p.add_argument("--feeds",     nargs="+", help="Override RSS feed URLs.")
    p.add_argument("--output",    type=str,  default="output", help="Output directory.")
    p.add_argument("--workers",   type=int,  default=2,        help="Parallel feed workers.")
    p.add_argument("--verbose",   action="store_true",         help="Debug logging.")
    p.add_argument("--dry-run",   action="store_true",         help="Parse feeds but don't save.")
    return p


# ─────────────────────────────────────────────
#  Entry point — callable from CLI or notebook
# ─────────────────────────────────────────────

def run(
    config:   str  = None,
    keywords: list = None,
    feeds:    list = None,
    output:   str  = "output",
    workers:  int  = 2,
    verbose:  bool = False,
    dry_run:  bool = False,
) -> None:
    \"\"\"
    Programmatic entry point — use this in Colab cells instead of CLI.
    All parameters mirror the CLI flags.
    \"\"\"
    setup_logging(verbose, log_dir=output)
    log = logging.getLogger("main")

    # ── Load config ───────────────────────────────────────────────
    if config:
        cfg = ScraperConfig.from_json(config)
        log.info("Loaded config from %s", config)
    else:
        cfg = ScraperConfig()

    # ── Override with explicit args ───────────────────────────────
    if keywords: cfg.keywords    = keywords
    if feeds:    cfg.feeds       = feeds
    cfg.output_dir  = output
    cfg.max_workers = workers

    log.info("=" * 60)
    log.info("NEWS SCRAPER starting")
    log.info("Keywords : %s", ", ".join(cfg.keywords))
    log.info("Feeds    : %d configured", len(cfg.feeds))
    log.info("Workers  : %d", cfg.max_workers)
    log.info("=" * 60)

    session = scrape(cfg)

    summary = session.summary()
    log.info("")
    log.info("=" * 60)
    log.info("SCRAPE COMPLETE — %d articles found", summary["total_articles"])
    for feed_url, count in summary["articles_by_feed"].items():
        log.info("  %-55s %d", feed_url, count)
    if summary["errors"]:
        log.warning("Feed errors:")
        for err in summary["errors"]:
            log.warning("  %s → %s", err["feed"], err["error"])
    log.info("")
    log.info("Top articles by relevance:")
    for item in summary["top_articles"]:
        log.info("  [%.2f] %s", item["score"], item["title"])
    log.info("=" * 60)

    if not dry_run:
        written = write_outputs(session, cfg)
        log.info("")
        log.info("Output files:")
        for label, path in written.items():
            log.info("  %-10s → %s", label, path)
    else:
        log.info("Dry run — no files written.")

    return session


def main() -> None:
    # parse_known_args ignores Colab's extra argv flags (--ip, --stdin, etc.)
    args, _ = build_parser().parse_known_args()
    run(
        config   = args.config,
        keywords = args.keywords,
        feeds    = args.feeds,
        output   = args.output,
        workers  = args.workers,
        verbose  = args.verbose,
        dry_run  = args.dry_run,
    )


if __name__ == "__main__":
    main()
"""
with open('/content/news_scraper/__main__.py', 'w', encoding='utf-8') as _f:
    _f.write(_src)
print('wrote __main__.py')


#### Write `analysis.py`


In [ ]:
# Auto-generated: writes analysis.py into the Colab filesystem
_src = """\"\"\"
news_scraper/analysis.py
------------------------
Text analysis module: frequency counting, phrase matching, and structured results.
Reads from the scraper's JSONL output.

Usage (CLI):
    python analysis.py
    python analysis.py --input output/index/master_index.jsonl
    python analysis.py --enrich   # also writes per-article enriched JSONL

Usage (Colab / notebook):
    from analysis import load_articles_from_jsonl, analyze_corpus, print_report
    articles = load_articles_from_jsonl(Path("output/index/master_index.jsonl"))
    results  = analyze_corpus(articles)
    print_report(results)
\"\"\"
from __future__ import annotations

import json
import re
import logging
import sys
import os
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional

def _ensure_path():
    import sys, os
    pkg = os.path.dirname(os.path.abspath(__file__))
    if pkg not in sys.path:
        sys.path.insert(0, pkg)

_ensure_path()

logger = logging.getLogger(__name__)


# ─────────────────────────────────────────────
#  Stopwords — comprehensive, no NLTK needed
# ─────────────────────────────────────────────

STOPWORDS: set[str] = {
    "a", "an", "the", "and", "or", "but", "if", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "as", "is", "was", "are", "were", "be", "been",
    "being", "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "not", "no", "nor", "so", "yet",
    "both", "either", "neither", "than", "that", "this", "these", "those", "it",
    "its", "we", "he", "she", "they", "them", "their", "our", "your", "my", "his",
    "her", "who", "which", "what", "when", "where", "how", "all", "each", "every",
    "any", "some", "such", "into", "through", "about", "between", "after", "before",
    "said", "says", "also", "more", "other", "new", "one", "two", "first", "last",
    "over", "under", "up", "out", "i", "you", "me", "him", "us", "am", "s", "re",
    "just", "been", "like", "very", "even", "well", "back", "still", "way", "get",
    "make", "go", "come", "take", "see", "know", "think", "use", "give", "most",
    "own", "while", "however", "since", "including", "according", "among", "against",
}


# ─────────────────────────────────────────────
#  Keyword taxonomy — multi-word phrases
# ─────────────────────────────────────────────

KEYWORDS_TECH: list[str] = [
    # missiles & rockets
    "ballistic missile", "cruise missile", "hypersonic missile",
    "anti-ship missile", "surface-to-air missile", "rocket launcher",
    # aircraft
    "fighter jet", "stealth aircraft", "stealth bomber",
    "attack helicopter", "armed drone", "combat drone", "uav", "ucav",
    # air defence
    "air defense", "air defence", "missile defense",
    "iron dome", "patriot system", "s-300", "s-400",
    # munitions
    "precision munition", "guided bomb", "bunker buster", "cluster munition",
    # naval
    "aircraft carrier", "naval strike", "submarine",
    # other
    "electronic warfare", "cyber attack", "interceptor",
]

KEYWORDS_LEADERS: list[str] = [
    # Iran
    "ali khamenei", "mojtaba khamenei", "masoud pezeshkian",
    "abbas araghchi", "mohammad pakpour", "esmail qaani",
    # US
    "donald trump", "pete hegseth", "marco rubio",
    "lindsey graham", "mike waltz",
    # Israel
    "benjamin netanyahu", "yoav gallant", "benny gantz",
    # others
    "mark carney", "emmanuel macron", "tamim bin hamad",
    "vladimir putin", "xi jinping", "olaf scholz", "rishi sunak",
]

KEYWORDS_ORGS: list[str] = [
    "irgc", "revolutionary guard", "quds force",
    "hezbollah", "hamas", "islamic jihad",
    "idf", "mossad", "cia", "pentagon",
    "nato", "un security council", "iaea", "centcom",
]

KEYWORDS_EVENTS: list[str] = [
    "airstrike", "air strike", "ceasefire", "cease-fire",
    "nuclear deal", "nuclear talks", "sanctions", "arms deal",
    "peace talks", "military exercise", "naval blockade",
    "ground invasion", "drone attack", "missile strike",
    "assassination", "prisoner exchange",
]


# ─────────────────────────────────────────────
#  I/O helpers
# ─────────────────────────────────────────────

def load_articles_from_jsonl(jsonl_path: Path) -> list[dict]:
    \"\"\"Load all article records from a JSONL file.\"\"\"
    articles = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                articles.append(json.loads(line))
            except json.JSONDecodeError as e:
                logger.warning("Skipping malformed line %d: %s", line_no, e)
    logger.info("Loaded %d articles from %s", len(articles), jsonl_path)
    return articles


def find_latest_jsonl(output_dir: Path = Path("output")) -> Optional[Path]:
    \"\"\"Auto-discover the most recent JSONL — master index first, then latest session.\"\"\"
    master = output_dir / "index" / "master_index.jsonl"
    if master.exists():
        return master
    sessions = sorted(
        (output_dir / "sessions").glob("*/articles.jsonl"),
        key=lambda p: p.parent.name,
        reverse=True,
    )
    return sessions[0] if sessions else None


def combine_text(articles: list[dict]) -> str:
    \"\"\"Concatenate title + full_text from all articles.\"\"\"
    parts = []
    for art in articles:
        title = art.get("title", "")
        body  = art.get("full_text", "") or art.get("summary", "")
        if title: parts.append(title)
        if body:  parts.append(body)
    return " ".join(parts)


# ─────────────────────────────────────────────
#  Analysis functions
# ─────────────────────────────────────────────

def tokenize(text: str, stopwords: set[str] = STOPWORDS) -> list[str]:
    \"\"\"Lowercase, strip punctuation, remove stopwords, drop 1-char tokens.\"\"\"
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", "", text)
    return [t for t in text.split() if t not in stopwords and len(t) > 1]


def count_phrases(text: str, phrases: list[str]) -> list[tuple[str, int]]:
    \"\"\"
    Case-insensitive substring count for each phrase.
    Returns only phrases with count > 0, sorted descending.
    \"\"\"
    text_lower = text.lower()
    counts = {
        phrase: text_lower.count(phrase.lower())
        for phrase in phrases
        if text_lower.count(phrase.lower()) > 0
    }
    return sorted(counts.items(), key=lambda x: x[1], reverse=True)


def top_unigrams(text: str, n: int = 30) -> list[tuple[str, int]]:
    \"\"\"Most frequent single words after stopword removal.\"\"\"
    return Counter(tokenize(text)).most_common(n)


# ─────────────────────────────────────────────
#  Per-article enrichment — for indexation
# ─────────────────────────────────────────────

def enrich_article(article: dict) -> dict:
    \"\"\"
    Add an analysis{} block to a single article dict in-place.
    Safe to call on any ArticleRecord dict from the scraper.
    \"\"\"
    text = (article.get("full_text", "") or "") + " " + (article.get("summary", "") or "")
    article["analysis"] = {
        "tech_mentions":   dict(count_phrases(text, KEYWORDS_TECH)),
        "leader_mentions": dict(count_phrases(text, KEYWORDS_LEADERS)),
        "org_mentions":    dict(count_phrases(text, KEYWORDS_ORGS)),
        "event_mentions":  dict(count_phrases(text, KEYWORDS_EVENTS)),
        "analyzed_at":     datetime.now(timezone.utc).isoformat(),
    }
    return article


# ─────────────────────────────────────────────
#  Corpus-level analysis
# ─────────────────────────────────────────────

def analyze_corpus(articles: list[dict]) -> dict:
    \"\"\"
    Full frequency analysis over all articles.
    Returns a structured dict suitable for JSON serialisation or DataFrame loading.
    \"\"\"
    text = combine_text(articles)
    return {
        "corpus_stats": {
            "total_articles": len(articles),
            "total_words":    len(text.split()),
            "analyzed_at":    datetime.now(timezone.utc).isoformat(),
        },
        "military_tech":     count_phrases(text, KEYWORDS_TECH)[:20],
        "political_leaders": count_phrases(text, KEYWORDS_LEADERS)[:20],
        "organizations":     count_phrases(text, KEYWORDS_ORGS)[:20],
        "event_types":       count_phrases(text, KEYWORDS_EVENTS)[:20],
        "top_unigrams":      top_unigrams(text, n=30),
    }


# ─────────────────────────────────────────────
#  Output
# ─────────────────────────────────────────────

def save_analysis(results: dict, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    logger.info("Analysis saved → %s", output_path)


def print_report(results: dict) -> None:
    cs = results["corpus_stats"]
    print(f"\\n{'='*55}")
    print(f"  CORPUS: {cs['total_articles']} articles  |  {cs['total_words']:,} words")
    print(f"{'='*55}")
    sections = [
        ("Military technology",   "military_tech"),
        ("Political leaders",     "political_leaders"),
        ("Organizations",         "organizations"),
        ("Event types",           "event_types"),
    ]
    for title, key in sections:
        items = results.get(key, [])
        if not items:
            continue
        print(f"\\n  {title}")
        print(f"  {'-'*40}")
        for phrase, count in items[:15]:
            bar = "█" * min(count, 30)
            print(f"  {phrase:<30} {count:>5}  {bar}")
    print(f"\\n  Top words (after stopwords)")
    print(f"  {'-'*40}")
    for word, count in results.get("top_unigrams", [])[:20]:
        bar = "█" * min(count // 5, 30)
        print(f"  {word:<30} {count:>5}  {bar}")
    print()


# ─────────────────────────────────────────────
#  Entry point
# ─────────────────────────────────────────────

def main() -> None:
    import argparse
    logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

    parser = argparse.ArgumentParser(description="Analyse scraped news corpus.")
    parser.add_argument("--input",  type=str, help="Path to JSONL (auto-detected if omitted).")
    parser.add_argument("--output", type=str, default="output/analysis", help="Output dir.")
    parser.add_argument("--enrich", action="store_true",
                        help="Write per-article enriched JSONL.")
    # parse_known_args so this is safe to import in Colab
    args, _ = parser.parse_known_args()

    jsonl_path = Path(args.input) if args.input else find_latest_jsonl()
    if not jsonl_path:
        logger.error("No JSONL file found. Run the scraper first, or pass --input <path>.")
        return

    logger.info("Reading from %s", jsonl_path)
    articles = load_articles_from_jsonl(jsonl_path)
    if not articles:
        logger.error("No articles loaded — nothing to analyse.")
        return

    results = analyze_corpus(articles)
    print_report(results)

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    out_dir   = Path(args.output)
    save_analysis(results, out_dir / f"corpus_analysis_{timestamp}.json")

    if args.enrich:
        enriched_path = out_dir / f"enriched_{timestamp}.jsonl"
        enriched_path.parent.mkdir(parents=True, exist_ok=True)
        with open(enriched_path, "w", encoding="utf-8") as f:
            for art in articles:
                f.write(json.dumps(enrich_article(art), ensure_ascii=False) + "\\n")
        logger.info("Enriched JSONL saved → %s", enriched_path)


if __name__ == "__main__":
    main()
"""
with open('/content/news_scraper/analysis.py', 'w', encoding='utf-8') as _f:
    _f.write(_src)
print('wrote analysis.py')


In [ ]:
# 1.3 — Set Python path, verify all files present, then import
import sys, os

pkg = "/content/news_scraper"
os.makedirs(pkg, exist_ok=True)

# Always put at front — safe to re-run
if pkg in sys.path:
    sys.path.remove(pkg)
sys.path.insert(0, pkg)

# Guard: confirm every module exists before importing
expected = ["config.py", "models.py", "scraper.py", "writer.py", "analysis.py", "__main__.py"]
missing  = [f for f in expected if not os.path.exists(os.path.join(pkg, f))]
if missing:
    msg = "Missing files: " + str(missing) + ". Re-run the write cells above, then re-run this cell."
    raise RuntimeError(msg)

# Clear any previously cached broken module state
for mod in ["config", "models", "scraper", "writer", "analysis"]:
    sys.modules.pop(mod, None)

from config   import ScraperConfig
from models   import ArticleRecord, ScrapeSession
from scraper  import scrape
from writer   import write_outputs
from analysis import load_articles_from_jsonl, analyze_corpus, print_report

print("All modules imported successfully")
print("Path:", pkg)
print("Files:", os.listdir(pkg))


---
## 2 · Configuration
Edit keywords and feeds here. The defaults target Middle East / Iran / Israel security coverage.


In [ ]:
from config import ScraperConfig

cfg = ScraperConfig(
    keywords = [
        'war', 'Iran', 'IRGC', 'US', 'Israel',
        'missile', 'sanctions', 'nuclear', 'ceasefire', 'airstrike',
    ],

    feeds = [
        # Think-tanks
        'https://www.csis.org/rss.xml',
        'https://www.rand.org/pubs/commentary.xml',
        'https://carnegieendowment.org/rss',
        'https://www.brookings.edu/feed/?post_type=research',
        'https://www.chathamhouse.org/rss.xml',
        'https://www.iiss.org/rss',
        # Foreign policy journals
        'https://foreignaffairs.com/rss.xml',
        'https://foreignpolicy.com/feed',
        'https://warontherocks.com/feed',
        'https://iswresearch.blogspot.com/feeds/posts/default',
        # Wire / broadcast
        'https://feeds.bbci.co.uk/news/world/rss.xml',
        'https://feeds.reuters.com/reuters/worldNews',
        'https://apnews.com/rss/apf-intlnews',
        'https://www.theguardian.com/world/rss',
        # Regional
        'https://www.aljazeera.com/xml/rss/all.xml',
        'https://www.france24.com/en/rss',
        'https://feeds.dw.com/rw/rss/rss.xml',
        # Defence
        'https://www.spacewar.com/Military_Technology.xml',
        'https://www.defense.gov/DesktopModules/ArticleCS/RSS.ashx?ContentType=1&Site=945&max=10',
    ],

    max_entries_per_feed   = 20,    # keep low for a fast trial run
    delay_between_requests = 1.5,
    request_timeout        = 12,
    max_workers            = 2,     # Colab free tier: 2 is safe
    output_dir             = '/content/output',
    min_word_count         = 80,
)

print(f'Config ready — {len(cfg.feeds)} feeds, {len(cfg.keywords)} keywords')


---
## 3 · Run the scraper
Fetches all RSS feeds in parallel, downloads matching articles, and saves structured output to `/content/output/`.

> **Expected time:** 3–10 minutes depending on keyword matches.


In [ ]:
import logging
logging.basicConfig(
    level   = logging.INFO,
    format  = '%(asctime)s  %(levelname)-8s  %(name)s — %(message)s',
    handlers= [logging.StreamHandler()],
    force   = True,
)

from scraper import scrape
from writer  import write_outputs

session = scrape(cfg)
summary = session.summary()

print(f'\n✅ Scrape complete — {summary["total_articles"]} articles')
print('\nArticles per feed:')
for feed, count in summary['articles_by_feed'].items():
    if count > 0:
        print(f'  {count:3d}  {feed}')

if summary['errors']:
    print('\n⚠️  Feed errors:')
    for e in summary['errors']:
        print(f'  {e["feed"]} → {e["error"]}')


In [ ]:
written = write_outputs(session, cfg)
print('Output files:')
for label, path in written.items():
    print(f'  {label:<10} → {path}')


---
## 4 · Inspect raw output


In [ ]:
# Top articles by relevance score
top = sorted(session.articles, key=lambda a: a.relevance_score, reverse=True)[:10]

print(f"{'#':<4} {'Score':<7} {'Words':<7} {'Lang':<6} Title")
print('-' * 80)
for i, art in enumerate(top, 1):
    print(f'{i:<4} {art.relevance_score:<7.3f} {art.word_count:<7} {art.language:<6} {art.title[:60]}')


In [ ]:
# Full inspection of one article — change ARTICLE_INDEX to browse
ARTICLE_INDEX = 0

art = top[ARTICLE_INDEX]
print(f'Title    : {art.title}')
print(f'URL      : {art.url}')
print(f'Feed     : {art.source_feed}')
print(f'Authors  : {art.authors}')
print(f'Published: {art.publish_date}')
print(f'Scraped  : {art.scraped_at}')
print(f'Words    : {art.word_count}   Score: {art.relevance_score}')
print(f'Matched keywords  : {art.matched_keywords}')
print(f'Extracted keywords: {art.extracted_keywords[:10]}')
print(f'\nSummary:\n{art.summary}')
print(f'\nFull text preview:\n{art.full_text[:600]}...')


In [ ]:
# JSONL round-trip check
import json
jsonl_path = written.get('jsonl')
if jsonl_path:
    with open(jsonl_path) as f:
        loaded = [json.loads(l) for l in f if l.strip()]
    print(f'JSONL round-trip: {len(loaded)} articles')
    print('Fields:', list(loaded[0].keys()))
else:
    print('No JSONL written — check config.save_jsonl')


---
## 5 · Corpus analysis


In [ ]:
from analysis import load_articles_from_jsonl, analyze_corpus, print_report, save_analysis
from pathlib import Path
from datetime import datetime, timezone

jsonl_path = written.get('index') or written.get('jsonl')
articles   = load_articles_from_jsonl(jsonl_path)
results    = analyze_corpus(articles)
print_report(results)


In [ ]:
ts       = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
out_path = Path('/content/output/analysis') / f'corpus_analysis_{ts}.json'
save_analysis(results, out_path)
print(f'Saved → {out_path}')


In [ ]:
# Optional pandas view
try:
    import pandas as pd
    for section in ['military_tech', 'political_leaders', 'organizations', 'event_types']:
        data = results.get(section, [])
        if data:
            df = pd.DataFrame(data, columns=['phrase', 'count'])
            print(f'\n── {section} ──')
            print(df.to_string(index=False))
except ImportError:
    print('pandas not available')


---
## 6 · Per-article enrichment
Adds an `analysis{}` block to each article — ready for indexation or ML.


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from analysis import enrich_article

ts            = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
enriched_path = Path('/content/output/analysis') / f'enriched_{ts}.jsonl'
enriched_path.parent.mkdir(parents=True, exist_ok=True)

with open(enriched_path, 'w', encoding='utf-8') as f:
    for art in articles:
        enriched = enrich_article(dict(art))
        f.write(json.dumps(enriched, ensure_ascii=False) + '\n')

print(f'✅ Enriched {len(articles)} articles → {enriched_path}')

# Preview analysis block of first article
with open(enriched_path) as f:
    sample = json.loads(f.readline())
print('\nanalysis{} block:')
print(json.dumps(sample.get('analysis', {}), indent=2))


---
## 7 · Export
Download all output files to your local machine.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/news_output', 'zip', '/content/output')
print('Archive ready: /content/news_output.zip')
files.download('/content/news_output.zip')


In [ ]:
# ── Optional: save to Google Drive (persists across sessions) ────────────
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree('/content/output', '/content/drive/MyDrive/news_scraper_output',
#                 dirs_exist_ok=True)
# print('Copied to Google Drive')
